# 药品说明书智能识别与语音播报系统

## 项目说明

针对药品说明书字体太小、老年人看不清读不懂的问题，本项目通过以下三个步骤，将药品说明书中的重点内容识别提取并语音播报：

1. **OCR 识别**：使用 PaddleOCR-VL-1.5 模型对药品说明书图片进行文字识别
2. **大模型整理**：使用 Qwen3-VL 多模态大模型对识别的文字进行整理，提取关键信息
3. **语音合成播报**：使用 Qwen3-TTS 语音合成模型将整理后的文字转为音频文件

### 提取的关键信息包括：
1. 药品名称
2. 药品适应症
3. 药品的用法与用量
4. 药品的禁忌
5. 药品的不良反应

### 技术栈：
- OCR: PaddleOCR-VL-1.5 + OpenVINO
- VLM: Qwen3-VL-4B-Instruct + OpenVINO
- TTS: Qwen3-TTS-CustomVoice-0.6B + OpenVINO

### 内存优化：
三个模型同时加载会占用大量内存。为避免一次性加载导致内存不足，本系统采用**按需加载**策略：
- 管线运行时，仅在需要某模型时才加载
- 每个步骤完成后自动释放上一个模型的内存
- 例如：OCR 完成后释放 OCR 模型，再加载 VLM 模型

#### 目录：
- [选择推理设备](#选择推理设备)
- [模型下载与检查](#模型下载与检查)
- [生成参数设置](#生成参数设置)
- [主流程](#主流程)
- [Gradio 交互界面](#Gradio-交互界面)

说明：此 notebook 依赖 https://github.com/megemini/medical_pipeline.git 中的代码，需要先 clone 到本地。

In [ ]:
import os

if not os.path.exists('gradio_helper.py') or not os.path.exists('requirements.txt'):
    !git clone https://github.com/megemini/medical_pipeline.git
    %cd medical_pipeline

!pip install -r requirements.txt

## 选择推理设备
[返回目录 ⬆️](#目录：)

选择用于推理的设备。CPU 兼容性最好，GPU 可在支持的硬件上提供加速。

In [ ]:
from notebook_utils import device_widget

device = device_widget(default="AUTO", exclude=["NPU"])
device

## 模型下载与检查
[返回目录 ⬆️](#目录：)

从 ModelScope 下载三个模型（如果已存在则跳过），并检查模型文件是否完整。

> **注意**：此步骤仅下载和检查模型，**不加载模型到内存**。模型将在管线运行时按需加载。

In [ ]:
from pathlib import Path

# --- OCR 模型 ---
ocr_model_dir = Path("PaddleOCR-VL-1.5-OpenVINO")

if not ocr_model_dir.exists():
    from modelscope import snapshot_download
    snapshot_download("megemini/PaddleOCR-VL-1.5-OpenVINO", local_dir=str(ocr_model_dir))
    print(f"OCR 模型已下载到: {ocr_model_dir}")
else:
    print(f"OCR 模型已存在: {ocr_model_dir}，跳过下载")

# --- VLM 模型 ---
vlm_model_dir = Path("Qwen3-VL-4B-Instruct-int4-ov")

if not vlm_model_dir.exists():
    from modelscope import snapshot_download
    snapshot_download("snake7gun/Qwen3-VL-4B-Instruct-int4-ov", local_dir=str(vlm_model_dir))
    print(f"VLM 模型已下载到: {vlm_model_dir}")
else:
    print(f"VLM 模型已存在: {vlm_model_dir}，跳过下载")

# --- TTS 模型 ---
tts_model_dir = Path("Qwen3-TTS-CustomVoice-0.6B-fp16-ov")

if not tts_model_dir.exists():
    from modelscope import snapshot_download
    snapshot_download("snake7gun/Qwen3-TTS-CustomVoice-0.6B-fp16-ov", local_dir=str(tts_model_dir))
    print(f"TTS 模型已下载到: {tts_model_dir}")
else:
    print(f"TTS 模型已存在: {tts_model_dir}，跳过下载")

In [ ]:
# ==================== 检查模型文件完整性 ====================

def check_model_files(model_dir, required_extensions):
    """检查模型目录中是否包含所需扩展名的文件"""
    if not model_dir.exists():
        return False
    found = set()
    for f in model_dir.iterdir():
        if f.is_file():
            found.add(f.suffix)
    missing = [ext for ext in required_extensions if ext not in found]
    if missing:
        print(f"  ⚠️ {model_dir}: 缺少扩展名 {missing} 的文件")
        return False
    print(f"  ✅ {model_dir}: 文件完整")
    return True

print("检查模型文件完整性...")
all_ok = True
all_ok &= check_model_files(ocr_model_dir, ['.xml', '.bin'])
all_ok &= check_model_files(vlm_model_dir, ['.xml', '.bin'])
all_ok &= check_model_files(tts_model_dir, ['.xml', '.bin'])

if all_ok:
    print("\n✅ 所有模型文件完整，可以开始运行管线")
else:
    print("\n⚠️ 部分模型文件不完整，请重新下载")

## 生成参数设置
[返回目录 ⬆️](#目录：)

设置三个模型的 `max_new_tokens` 参数，控制每个模型生成的最大 token 数量：
- **OCR max_new_tokens**：PaddleOCR-VL 识别文字时的最大生成长度，说明书内容多时建议调大
- **VLM max_new_tokens**：Qwen3-VL 提取信息时的最大生成长度，需要更详细整理时可调大
- **TTS max_new_tokens**：Qwen3-TTS 语音合成时的最大生成长度，文本较长时建议调大

In [ ]:
# ==================== 生成参数设置 ====================

# OCR 最大生成 token 数（说明书内容多时建议调大，默认 5120）
ocr_max_new_tokens = 5120

# VLM 最大生成 token 数（需要更详细整理时可调大，默认 1024）
vlm_max_new_tokens = 1024

# TTS 最大生成 token 数（文本较长时建议调大，默认 2048）
tts_max_new_tokens = 2048

print(f"OCR max_new_tokens: {ocr_max_new_tokens}")
print(f"VLM max_new_tokens: {vlm_max_new_tokens}")
print(f"TTS max_new_tokens: {tts_max_new_tokens}")

## 主流程
[返回目录 ⬆️](#目录：)

主流程包含以下步骤：
1. 加载图片
2. 图片分割（可选，针对文字太小的说明书，将图片切割成多部分进行识别，分割的图片有重叠）
3. OCR 文字识别（**按需加载 OCR 模型，完成后释放**）
4. 大模型文字整理（**按需加载 VLM 模型，完成后释放**）
5. 语音合成（**按需加载 TTS 模型，完成后释放**）

> 通过 `ModelManager` 实现按需加载，每个步骤只加载当前需要的模型，步骤完成后释放内存，避免三个模型同时占用 6-7 GB 内存。

In [ ]:
# ==================== 配置日志 ====================
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(name)s] %(levelname)s: %(message)s",
    datefmt="%H:%M:%S",
)

print("✅ 日志配置完成 (级别: INFO)")

In [ ]:
# ==================== 初始化 ModelManager ====================
from gradio_helper import ModelManager, drug_ocr_pipeline

model_manager = ModelManager(
    ocr_model_dir=ocr_model_dir,
    vlm_model_dir=vlm_model_dir,
    tts_model_dir=tts_model_dir,
    device=device.value,
)

print("✅ ModelManager 初始化完成（模型尚未加载，将在管线运行时按需加载）")

In [ ]:
# ==================== 运行主流程示例 ====================

sample_image_path = str(Path("1.jpg"))

result = drug_ocr_pipeline(
    model_manager=model_manager,
    image_path=sample_image_path,
    enable_split=True,
    num_splits=4,
    overlap_ratio=0.1,
    ocr_max_new_tokens=ocr_max_new_tokens,
    vlm_max_new_tokens=vlm_max_new_tokens,
    tts_max_new_tokens=tts_max_new_tokens,
    release_between_steps=True,  # 每步完成后释放模型内存
)

print("\n" + "=" * 60)
print("📋 OCR 识别结果:")
print("=" * 60)
print(result["ocr_text"][:500] + "..." if len(result["ocr_text"]) > 500 else result["ocr_text"])

print("\n" + "=" * 60)
print("📝 大模型整理结果:")
print("=" * 60)
print(result["extracted_info"])

# 播放音频
if result["audio"] is not None:
    import IPython.display as ipd
    sr, wav_data = result["audio"]
    print("\n🔊 播放语音...")
    ipd.display(ipd.Audio(wav_data, rate=sr))

## Gradio 交互界面
[返回目录 ⬆️](#目录：)

通过 Gradio 界面，用户可以：
- 上传药品说明书图片
- 设置是否启用图片分割及分割数量
- 调整各模型的生成参数（max_new_tokens）
- 选择 TTS 说话人和语言
- 查看识别和整理结果
- 播放语音合成的音频

> 模型在每次点击"开始识别"时按需加载，步骤间自动释放内存。

In [ ]:
from gradio_helper import make_demo

demo = make_demo(
    model_manager=model_manager,
    ocr_max_new_tokens=ocr_max_new_tokens,
    vlm_max_new_tokens=vlm_max_new_tokens,
    tts_max_new_tokens=tts_max_new_tokens,
)

# server_name="0.0.0.0" 使 Gradio 监听所有网络接口，
# 解决 Jupyter 环境中 127.0.0.1 refused to connect 的问题。
# 如果仍然无法访问，会自动回退到 share=True 创建公网隧道。
try:
    demo.launch(server_name="0.0.0.0", server_port=7860, debug=True)
except Exception:
    demo.launch(debug=True, share=True)